# Notebook 09 — When Embeddings Fool You: Five Gotchas (Lesson 13)

> **Easiest way to run this: Google Colab — nothing to install.**
> Go to https://colab.research.google.com then File > Upload notebook and choose this file.
> Prefer your own computer? Lesson 1 shows the VS Code and local-Jupyter paths.

Five honest failure modes, each demonstrated, each with its defence. One gotcha at a time.

In [ ]:
# Run once: model + the cosine helper from Lesson 5.
%pip install -q sentence-transformers
import math
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

def dot_product(a, b):
    return sum(x * y for x, y in zip(a, b))

def magnitude(v):
    return math.sqrt(sum(x * x for x in v))

def cosine_similarity(a, b):
    return dot_product(a, b) / (magnitude(a) * magnitude(b))

print("Ready.")

## Gotcha 1 — there is no single "sentence embedding"
A sentence vector is POOLED from word vectors, and the recipe matters. First: ask our model
what it does (the printout says `pooling_mode_mean_tokens: True` — mean pooling).

In [ ]:
# Activity: read the model's own pooling label.
print(model)

Now SEE why naive pooling can mislead, with tiny hand-built word vectors. Real
sentences are mostly filler ("the", "is", "on"). Averaging everything equally lets the
filler vote — two UNRELATED sentences come out looking nearly identical.

In [ ]:
# Activity: naive mean pooling vs weighted pooling, on readable toy vectors.
# dims: [animal, motion, tech, filler]
WORD = {
    "dog": [1, 0, 0, 0], "puppy": [0.95, 0, 0, 0.1], "runs": [0, 1, 0, 0],
    "sprints": [0, 0.95, 0, 0], "python": [0, 0, 1, 0], "code": [0, 0, 1, 0],
}
for stop in ["the", "a", "is", "on", "and", "it", "mat", "rug", "screen"]:
    WORD[stop] = [0, 0, 0, 1]                       # filler words sit on the filler axis
WEIGHT = {w: (0.05 if WORD[w] == [0, 0, 0, 1] else 1.0) for w in WORD}  # filler ~ free

def mean_pool(words):
    vs = [WORD[w] for w in words]
    return [sum(c) / len(vs) for c in zip(*vs)]

def weighted_pool(words):
    total = sum(WEIGHT[w] for w in words)
    return [sum(WEIGHT[w] * WORD[w][d] for w in words) / total for d in range(4)]

A = "the dog is on the mat and it runs".split()       # about a dog
B = "a puppy is on a rug and it sprints".split()      # paraphrase of A
C = "the python is on the screen and it is code".split()  # UNRELATED (tech)

print("naive mean   cos(A,B) paraphrase:", round(cosine_similarity(mean_pool(A), mean_pool(B)), 3))
print("naive mean   cos(A,C) UNRELATED :", round(cosine_similarity(mean_pool(A), mean_pool(C)), 3))
print("weighted     cos(A,B) paraphrase:", round(cosine_similarity(weighted_pool(A), weighted_pool(B)), 3))
print("weighted     cos(A,C) UNRELATED :", round(cosine_similarity(weighted_pool(A), weighted_pool(C)), 3))
# Naive mean scores the unrelated pair ~0.95 — the filler dominated.
# Weighting filler down restores honesty. Trained models like MiniLM learn this for you.

## Gotcha 2 — the silent cut-off (recap)
You proved this in Notebook 03: "pizza" past the 256-token mark scored 0.102 — never read.
Defence: chunk long documents. Re-run that cell there if you want to see it again.

## Gotcha 3 — rare names shatter
A made-up product code means nothing to the tokenizer; it turns to confetti, and vector
search goes mushy on it. Defence: a literal keyword search alongside the vectors.

In [ ]:
# Activity: tokenize a made-up product name, then watch keyword search save the day.
print("tokens:", model.tokenizer.tokenize("ZorpTab90"))

corpus = [
    "The ZorpTab90 phone costs 90 dollars.",
    "The laptop is on sale this week.",
    "Buy a brand new gadget today.",
]
# The defence: an exact keyword match, no embedding needed.
hits = [s for s in corpus if "zorptab90" in s.lower()]
print("keyword search for 'zorptab90':", hits)

## Gotcha 4 — a similarity score is not a grade
Some models flatter everything (every pair 0.9+); ours scores unrelated pairs near 0.
Same threshold, opposite disasters. Trust the RANKING, calibrate per model.

In [ ]:
# Activity: a toy "flattering" model — every vector shares a big common component.
BIAS = [2.0, 2.0, 2.0]
CONTENT = {
    "cats are lazy pets":   [1.0, 0.0, 0.0],
    "dogs love long walks": [0.9, 0.0, 0.0],
    "python powers ai":     [0.0, 1.0, 0.0],
    "pasta needs basil":    [0.0, 0.0, 1.0],
}
EMB = {t: [c + b for c, b in zip(v, BIAS)] for t, v in CONTENT.items()}

texts = list(EMB)
for i, t1 in enumerate(texts):
    for t2 in texts[i + 1:]:
        print(f"cos = {cosine_similarity(EMB[t1], EMB[t2]):.2f}   {t1!r} vs {t2!r}")
# EVERYTHING scores ~0.9+, even pasta vs python. An 'above 0.8' rule keeps it all.

In [ ]:
# Activity: our REAL model is the opposite — unrelated pairs score near zero.
pairs = [
    ("cats are lazy pets", "pasta needs basil"),
    ("python powers ai", "dogs love long walks"),
    ("the meeting is on tuesday", "bananas are yellow"),
]
for x, y in pairs:
    print(f"cos = {cosine_similarity(model.encode(x), model.encode(y)):.3f}   {x!r} vs {y!r}")
# Near 0.0 here. So 'above 0.8' would throw EVERYTHING away on this model.
# Same rule, opposite failure: raw scores do not transfer between models.

## Gotcha 5 — embeddings barely see "not" or numbers
Opposite policies score 0.915. A 2-day and a 20-day promise score 0.904. Defences: let a
reader read the retrieved text; pull numbers into structured fields.

In [ ]:
# Activity: the negation trap, on the real model.
A = model.encode("Returns are accepted within 30 days.")
B = model.encode("Returns are not accepted within 30 days.")
q = model.encode("are returns accepted?")

print("cos(A, B)  opposite meanings:", round(cosine_similarity(A, B), 3))   # 0.915
print("query vs A (accepted)       :", round(cosine_similarity(q, A), 3))   # 0.776
print("query vs B (NOT accepted)   :", round(cosine_similarity(q, B), 3))   # 0.715

In [ ]:
# Activity: the numbers trap, and the structured-field fix.
import re

a = "Your order ships within 2 days."
b = "Your order ships within 20 days."
print("cos(2-day, 20-day):", round(cosine_similarity(model.encode(a), model.encode(b)), 3))  # 0.904

# The fix: parse the number OUT of the text and compare it as a number.
def days(text):
    match = re.search(r"(\d+)\s*days?", text)
    return int(match.group(1)) if match else None

print("parsed:", days(a), "vs", days(b), "-> equal?", days(a) == days(b))

Five gotchas, five defences: know your pooling, chunk long text, run hybrid search,
trust rankings not thresholds, and let a reader read. You now know what most tutorials
never teach.

## Practice — Your Turn

Three short exercises. Each one is a gotcha you can trigger yourself. Before you run the answer, write down what you expect to happen. These traps fool almost everyone the first time, so the prediction is half the lesson.

All answers use the `model` and the `cosine_similarity` helper you loaded at the top.

### Exercise 1 — Write a sentence and its opposite

Pick a plain statement and flip its meaning by adding the word `not`. For example, "The store is open on Sundays." versus "The store is not open on Sundays." These two sentences mean opposite things to a human reader.

Encode both with the model and print their cosine similarity. **Predict first:** on a scale from 0 (unrelated) to 1 (identical), what score do you expect for two sentences that say opposite things?

Try it yourself, then run the answer cell below.

In [ ]:
# Answer
open_sun = "The store is open on Sundays."            # the plain statement
closed_sun = "The store is not open on Sundays."      # the opposite, one word changed

v_open = model.encode(open_sun)                       # vector for the open version
v_closed = model.encode(closed_sun)                   # vector for the not-open version

score = cosine_similarity(v_open, v_closed)           # compare the two meanings
print("cosine of opposite sentences:", round(score, 3))  # comes out about 0.9

# The score lands near 0.9, very high, even though the meanings are reversed.
# The model barely registers the word 'not'. The two sentences share almost
# every word, so they look nearly the same to it.
# Defence: do not let a score alone decide. Let a reader read the retrieved
# sentence and confirm the store is actually open.

### Exercise 2 — Change only the number

Now keep every word the same and change just a number. Compare "The package weighs 2 kilograms." with "The package weighs 20 kilograms." A 2 kg parcel and a 20 kg parcel are very different in the real world.

Encode both and print their cosine similarity. **Predict first:** will the model treat 2 and 20 as far apart, or will it smooth over them?

Try it yourself, then run the answer cell below.

In [ ]:
# Answer
light = "The package weighs 2 kilograms."             # the 2 kg version
heavy = "The package weighs 20 kilograms."            # the 20 kg version

v_light = model.encode(light)                         # vector for 2 kg
v_heavy = model.encode(heavy)                         # vector for 20 kg

score = cosine_similarity(v_light, v_heavy)           # compare the two weights
print("cosine of 2 kg vs 20 kg:", round(score, 3))   # comes out about 0.9

# The score is high, around 0.9. The model smooths over the exact figure and
# sees two near-identical sentences about a package weight.
# Defence: keep important numbers in a structured field (a real number column)
# and compare them as numbers, not as text inside a sentence.

### Exercise 3 — Why "above 0.8" is a model-specific rule

A common shortcut is "keep every match above 0.8 and throw the rest away." The trouble is that the number 0.8 means different things on different models. Take two unrelated sentences on our model and see where they actually land.

Encode an unrelated pair and print the cosine. **Predict first:** for two sentences with nothing in common, do you expect a flat 0, or something a bit above 0?

Try it yourself, then run the answer cell below.

In [ ]:
# Answer
s1 = "The meeting is scheduled for Tuesday morning."  # one topic: a meeting
s2 = "Bananas turn brown when they are ripe."         # an unrelated topic: fruit

v1 = model.encode(s1)                                 # vector for the meeting sentence
v2 = model.encode(s2)                                 # vector for the banana sentence

score = cosine_similarity(v1, v2)                     # how similar does the model think they are
print("cosine of two unrelated sentences:", round(score, 3))  # a small positive number, not a clean 0

# The score is low but not zero. Unrelated text on this model still sits a little
# above 0, and a flattering model would put the same pair near 0.9 (see Gotcha 4).
# So a fixed rule like 'above 0.8' is tuned to one model and breaks on another.
# Defence: trust the ranking (which match is closest), and calibrate any cut-off
# on the model you are actually using.